# Library Landscape: Unified Forecasting with sktime

**Docker image**: `ml4t-gpu`

This notebook compares raw PyTorch implementations against library-wrapped
alternatives, positioning **sktime as the unifying meta-framework** for time
series forecasting. sktime provides a consistent `fit()`/`predict()` API that
wraps NeuralForecast, PyTorch Forecasting, HuggingFace, and more -- letting
practitioners swap architectures without rewriting data pipelines.

**Learning Objectives**:
- Compare raw PyTorch vs library-wrapped implementations
- Understand sktime's unified fit/predict API for time series
- See how sktime wraps NeuralForecast, PyTorch Forecasting, and HuggingFace
- Evaluate when to use raw PyTorch vs library abstractions

**Book Reference**: Chapter 13, Section 13.7 (A Practitioner's Framework)

**Prerequisites**: ETF price data via the canonical `load_etfs()` loader

In [1]:
"""Library Landscape — compare raw PyTorch vs sktime-wrapped forecasting implementations."""

import tempfile
import time
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import polars as pl
import torch
import torch.nn as nn
from darts import TimeSeries
from darts.models import RNNModel
from ml4t.diagnostic.metrics import pooled_ic
from sktime.forecasting.chronos import ChronosForecaster
from sktime.forecasting.patch_tst import PatchTSTForecaster

from utils.reproducibility import set_global_seeds

warnings.filterwarnings("ignore")

from data import load_etfs

/opt/ml4t/lib/python3.14/site-packages/multiprocess/connection.py:335: SyntaxWarning: 'return' in a 'finally' block
  return f
/opt/ml4t/lib/python3.14/site-packages/multiprocess/connection.py:337: SyntaxWarning: 'return' in a 'finally' block
  return self._get_more_data(ov, maxsize)


In [2]:
SEED = 42
LOOKBACK = 60
HORIZON = 5
HIDDEN_SIZE = 64
EPOCHS = 30
BATCH_SIZE = 32
START_DATE = "2015-01-01"

In [3]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

set_global_seeds(SEED)

Device: cuda


## Data: SPY Close Prices

We use a single univariate series (SPY daily closes) so every library sees
identical input. We prepare two representations: NumPy sequences for raw
PyTorch, and a pandas Series with business-day index for sktime/Darts.

In [4]:
etf_df = load_etfs()
start_dt = datetime.fromisoformat(START_DATE)

spy = (
    etf_df.filter((pl.col("symbol") == "SPY") & (pl.col("timestamp") >= start_dt))
    .sort("timestamp")
    .select(["timestamp", "close"])
)

prices = spy["close"].to_numpy().astype(np.float32)
price_mean, price_std = prices.mean(), prices.std()
prices_norm = (prices - price_mean) / price_std

print(f"SPY: {len(prices):,} daily observations")

SPY: 2,766 daily observations


### PyTorch Sequences (Normalized)

In [5]:
def create_sequences(data, lookback, horizon):
    """Create (input, target) pairs for univariate forecasting."""
    X, y = [], []
    for i in range(len(data) - lookback - horizon + 1):
        X.append(data[i : i + lookback])
        y.append(data[i + lookback : i + lookback + horizon].mean())
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)


X, y = create_sequences(prices_norm, LOOKBACK, HORIZON)
split_idx = int(len(X) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

X_train_t = torch.FloatTensor(X_train).unsqueeze(-1).to(DEVICE)
y_train_t = torch.FloatTensor(y_train).unsqueeze(-1).to(DEVICE)
X_test_t = torch.FloatTensor(X_test).unsqueeze(-1).to(DEVICE)

print(f"Sequences: train={len(X_train):,}, test={len(X_test):,}")

Sequences: train=2,161, test=541


### sktime / Darts Series (Prices with DatetimeIndex)

In [6]:
spy_pd = spy.to_pandas()
spy_series = spy_pd.set_index("timestamp")["close"]
spy_series = spy_series.sort_index()
spy_series.index = pd.DatetimeIndex(spy_series.index)

# Reindex to business-day frequency, forward-filling gaps (holidays etc.)
# Libraries like sktime/NeuralForecast and Darts require a regular frequency.
bday_idx = pd.bdate_range(start=spy_series.index.min(), end=spy_series.index.max(), freq="B")
spy_series = spy_series.reindex(bday_idx).ffill().dropna()
spy_series.index.freq = "B"

sk_split = int(len(spy_series) * 0.8)
y_train_sk = spy_series.iloc[:sk_split]
y_test_sk = spy_series.iloc[sk_split : sk_split + HORIZON]
fh_sk = list(range(1, HORIZON + 1))

### Evaluation Helper

**Note on Spearman IC**: In this univariate setting, IC reduces to rank
correlation between predicted and actual values at consecutive time points —
not the cross-sectional rank correlation used in multi-asset notebooks (04–10).
We include it for metric consistency across notebooks, but MSE is the primary
metric here.

In [7]:
def evaluate(y_true, y_pred):
    """Compute MSE and Spearman IC."""
    mse = float(np.mean((y_true - y_pred) ** 2))
    ic = pooled_ic(y_pred, y_true) if len(y_true) > 5 else float("nan")
    return {"mse": round(mse, 6), "ic": round(ic, 4) if np.isfinite(ic) else None}


results = []  # Collector for final comparison

## Part 1: Raw PyTorch LSTM

The "full control" baseline: model definition, training loop, and evaluation
require roughly 50 lines of boilerplate.

In [8]:
class LSTMForecaster(nn.Module):
    """Minimal LSTM for univariate forecasting."""

    def __init__(self, hidden_size, num_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(1, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

In [9]:
model = LSTMForecaster(HIDDEN_SIZE).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

start = time.time()
for epoch in range(EPOCHS):
    model.train()
    indices = torch.randperm(len(X_train_t))
    epoch_loss, n_batches = 0.0, 0
    for i in range(0, len(X_train_t), BATCH_SIZE):
        batch_idx = indices[i : i + BATCH_SIZE]
        loss = criterion(model(X_train_t[batch_idx]), y_train_t[batch_idx])
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        n_batches += 1
    if (epoch + 1) % 10 == 0:
        print(f"  Epoch {epoch + 1}/{EPOCHS}: loss={epoch_loss / n_batches:.6f}")
pytorch_time = time.time() - start

model.eval()
with torch.no_grad():
    preds_pt = model(X_test_t).cpu().numpy().flatten()

m = evaluate(y_test, preds_pt)
results.append(
    {
        "Model": "LSTM",
        "Library": "Raw PyTorch",
        "Lines": "~50",
        "Train Time (s)": round(pytorch_time, 1),
        **m,
        "Trainable": True,
        "Target Scale": "normalized",
        "Evaluation Points": len(y_test),
    }
)
print(f"Raw PyTorch LSTM: {pytorch_time:.1f}s, MSE={m['mse']}, IC={m['ic']}")

  Epoch 10/30: loss=0.002107


  Epoch 20/30: loss=0.001787


  Epoch 30/30: loss=0.001581
Raw PyTorch LSTM: 3.7s, MSE=0.095823, IC=0.9888


The raw PyTorch LSTM requires defining the model class, training loop, and
evaluation — approximately 50 lines. Full control over architecture and
training dynamics, at the cost of significant boilerplate.

## Part 2: sktime + NeuralForecast LSTM

sktime wraps NeuralForecast's LSTM: three lines replace the 50-line training
loop. The underlying recurrent architecture is equivalent.

**Dependency note**: sktime's neural forecasters require `neuralforecast`, which
depends on `ray` — and Python 3.14 wheels are still pending in
([ray-project/ray#56434](https://github.com/ray-project/ray/issues/56434)).
Once those wheels land, install via `uv pip install neuralforecast`
and uncomment the demo below. Recent sktime releases also renamed
`hidden_size` to `encoder_hidden_size`.

In [10]:
# from sktime.forecasting.neuralforecast import NeuralForecastLSTM
#
# forecaster = NeuralForecastLSTM(
#     freq="B",
#     input_size=LOOKBACK,
#     max_steps=EPOCHS * 10,
#     encoder_hidden_size=HIDDEN_SIZE,
# )
# start = time.time()
# forecaster.fit(y_train_sk)
# y_pred = forecaster.predict(fh=fh_sk)
# t = time.time() - start
#
# m = evaluate(y_test_sk.values, y_pred.values)
# results.append(
#     {
#         "Model": "LSTM",
#         "Library": "sktime/NeuralForecast",
#         "Lines": "~5",
#         "Train Time (s)": round(t, 1),
#         **m,
#         "Trainable": True,
#     }
# )
# print(f"sktime NeuralForecastLSTM: {t:.1f}s, MSE={m['mse']}, IC={m['ic']}")

sktime reduces the same LSTM to three lines (`fit`/`predict`), delegating to
NeuralForecast's optimized training loop underneath.

## Part 3: sktime + PyTorch Forecasting N-BEATS

N-BEATS (Section 13.2) via sktime's PyTorch Forecasting wrapper. Same
`fit()`/`predict()` API, different architecture underneath.

**Dependency note**: sktime's PyTorch Forecasting wrapper requires
`pytorch-forecasting`. The previous run also tripped over a wrapper API
mismatch around `max_prediction_length`/`max_encoder_length`. Once the
wrapper signature stabilizes, install `pytorch-forecasting` and uncomment
the demo below.

In [11]:
# from sktime.forecasting.pytorchforecasting import PytorchForecastingNBeats
#
# forecaster = PytorchForecastingNBeats(
#     max_prediction_length=HORIZON,
#     max_encoder_length=LOOKBACK,
#     max_epochs=min(EPOCHS, 10),
#     trainer_kwargs={"enable_progress_bar": False, "accelerator": "auto"},
# )
# start = time.time()
# forecaster.fit(y_train_sk)
# y_pred = forecaster.predict(fh=fh_sk)
# t = time.time() - start
#
# m = evaluate(y_test_sk.values, y_pred.values)
# results.append(
#     {
#         "Model": "N-BEATS",
#         "Library": "sktime/PyTorchForecasting",
#         "Lines": "~5",
#         "Train Time (s)": round(t, 1),
#         **m,
#         "Trainable": True,
#     }
# )
# print(f"sktime N-BEATS: {t:.1f}s, MSE={m['mse']}, IC={m['ic']}")

Same three-line API, different architecture. Swapping LSTM for N-BEATS requires
changing only the import and constructor — no data pipeline changes.

## Part 4: sktime PatchTST

PatchTST (Section 13.5) via sktime's HuggingFace integration. Supports
full training, fine-tuning, and zero-shot modes.

**Dependency note**: PatchTST uses sktime's HuggingFace backend, not
`neuralforecast`, so it runs in the current Python 3.14 environment. The
first run downloads model artifacts and trains into a temporary output
directory.

In [12]:
forecaster = PatchTSTForecaster(
    fit_strategy="full",
    config={
        "context_length": LOOKBACK,
        "prediction_length": HORIZON,
        "patch_length": 12,
        "num_hidden_layers": 2,
        "d_model": HIDDEN_SIZE,
    },
    training_args={
        "output_dir": tempfile.mkdtemp(prefix="patchtst_"),
        "num_train_epochs": min(EPOCHS, 10),
        "per_device_train_batch_size": BATCH_SIZE,
        "logging_strategy": "no",
    },
)
start = time.time()
forecaster.fit(y_train_sk, fh=fh_sk)
y_pred = forecaster.predict()
t = time.time() - start

m = evaluate(y_test_sk.values, y_pred.values[:HORIZON])
results.append(
    {
        "Model": "PatchTST",
        "Library": "sktime/HuggingFace",
        "Lines": "~8",
        "Train Time (s)": round(t, 1),
        **m,
        "Trainable": True,
        "Target Scale": "raw_price",
        "Evaluation Points": HORIZON,
    }
)
print(f"sktime PatchTST: {t:.1f}s, MSE={m['mse']}, IC={m['ic']}")

Step,Training Loss


sktime PatchTST: 8.9s, MSE=270.813966, IC=None


PatchTST via HuggingFace requires slightly more configuration (patch size,
training arguments) but still follows the `fit`/`predict` pattern.

## Part 5: sktime Chronos (Zero-Shot Foundation Model)

Chronos (Section 13.6) requires **no training**. The `fit()` call registers
history; all computation happens in `predict()`. This is the ultimate
rapid-prototyping workflow: instant baseline with zero boilerplate.

**Dependency note**: Chronos also uses sktime's HuggingFace backend and runs
in the current Python 3.14 environment. The first run downloads the Chronos
checkpoint before generating zero-shot forecasts.

In [13]:
forecaster = ChronosForecaster("amazon/chronos-t5-tiny")
start = time.time()
forecaster.fit(y_train_sk)
y_pred = forecaster.predict(fh=fh_sk)
t = time.time() - start

m = evaluate(y_test_sk.values, y_pred.values)
results.append(
    {
        "Model": "Chronos (tiny)",
        "Library": "sktime/HuggingFace",
        "Lines": "~3",
        "Train Time (s)": round(t, 1),
        **m,
        "Trainable": False,
        "Target Scale": "raw_price",
        "Evaluation Points": HORIZON,
    }
)
print(f"sktime Chronos: {t:.1f}s (zero-shot), MSE={m['mse']}, IC={m['ic']}")

config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/33.6M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

sktime Chronos: 1.8s (zero-shot), MSE=68.218539, IC=None


Zero-shot inference eliminates training entirely. Accuracy depends on domain
match with the pretraining corpus (see Section 13.6 for foundation model details).

## Part 6: Darts LSTM (Comparison Point)

Darts is a popular alternative with its own `TimeSeries` container and API.
This provides one comparison point outside the sktime ecosystem.

In [14]:
# Use the reindexed business-day series to ensure freq="B" is consistent.
# Cast to float32: Darts TimeSeries default to float64, but the Lightning
# trainer's accelerator="auto" selects MPS on Apple Silicon, and MPS rejects
# float64 tensors. float32 runs identically on CPU, CUDA, and MPS.
spy_bday_df = spy_series.reset_index()
spy_bday_df.columns = ["timestamp", "close"]
ts = TimeSeries.from_dataframe(
    spy_bday_df, time_col="timestamp", value_cols="close", freq="B"
).astype(np.float32)
ts_train = ts[:sk_split]
ts_test = ts[sk_split : sk_split + HORIZON]

darts_model = RNNModel(
    model="LSTM",
    input_chunk_length=LOOKBACK,
    output_chunk_length=HORIZON,
    training_length=LOOKBACK + HORIZON,
    hidden_dim=HIDDEN_SIZE,
    n_epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    random_state=SEED,
    pl_trainer_kwargs={"enable_progress_bar": False, "accelerator": "auto"},
)
start = time.time()
darts_model.fit(ts_train)
y_pred_darts = darts_model.predict(HORIZON)
t = time.time() - start

m = evaluate(ts_test.values().flatten()[:HORIZON], y_pred_darts.values().flatten()[:HORIZON])
results.append(
    {
        "Model": "LSTM",
        "Library": "Darts",
        "Lines": "~10",
        "Train Time (s)": round(t, 1),
        **m,
        "Trainable": True,
        "Target Scale": "raw_price",
        "Evaluation Points": HORIZON,
    }
)
print(f"Darts LSTM: {t:.1f}s, MSE={m['mse']}, IC={m['ic']}")

ignoring user defined `output_chunk_length`. RNNModel uses a fixed `output_chunk_length=1`.


GPU available: True (cuda), used: True


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


You are using a CUDA device ('NVIDIA GeForce RTX 3090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision


...

GPU available: True (cuda), used: True


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Darts LSTM: 6.3s, MSE=102806.703125, IC=None


Darts provides a self-contained ecosystem with its own `TimeSeries` container.
The `random_state` parameter ensures reproducibility — a feature not all
wrapper APIs expose.

## Comparison Table

All approaches use the same data (SPY close prices). The table below shows
results from libraries that are currently runnable in this environment.
Four of the six demos execute end-to-end today: raw PyTorch, PatchTST,
Chronos, and Darts. The sktime NeuralForecast wrapper is blocked by the
`ray` dependency on Python 3.14, and the PyTorch Forecasting wrapper is
blocked by an upstream API mismatch.

**Comparison caveat**: the `MSE` column is not directly comparable across
rows. The `Target Scale` and `Evaluation Points` columns make the mismatch
explicit: raw PyTorch fits on a *normalized* target and evaluates on a long
rolling-window test set, while sktime PatchTST/Chronos and Darts operate on
raw price levels and evaluate on a single `HORIZON`-step forecast. Both
axes — scale and `n_eval` — break apples-to-apples comparison. Read the
table as "implementation experience", not as an accuracy benchmark.

In [15]:
if results:
    comparison = pl.DataFrame(results)
else:
    comparison = None
    print("No results collected -- check library installations.")

comparison

Model,Library,Lines,Train Time (s),mse,ic,Trainable,Target Scale,Evaluation Points
str,str,str,f64,f64,f64,bool,str,i64
"""LSTM""","""Raw PyTorch""","""~50""",3.7,0.095823,0.9888,true,"""normalized""",541
"""PatchTST""","""sktime/HuggingFace""","""~8""",8.9,270.813966,null,true,"""raw_price""",5
"""Chronos (tiny)""","""sktime/HuggingFace""","""~3""",1.8,68.218539,null,false,"""raw_price""",5
"""LSTM""","""Darts""","""~10""",6.3,102806.703125,null,true,"""raw_price""",5


## When to Use Each Approach

The section text (Section 13.7) compares library *capabilities* (panel support,
foundation models). This table focuses on *implementation experience*:

| Approach | Implementation Effort | Flexibility | Best For |
|----------|----------------------|-------------|----------|
| **Raw PyTorch** | ~50 lines, full boilerplate | Full control over everything | Custom architectures, cross-sectional ranking, research |
| **sktime** | ~3–8 lines | Constrained to `fit`/`predict` API | Rapid prototyping, backend swapping, standardized benchmarks |
| **Darts** | ~10 lines | Moderate (own `TimeSeries` container) | Probabilistic forecasting, self-contained ecosystem |
| **Chronos / TSFMs** | ~3 lines, zero training | None (zero-shot only) | Instant baselines, cold-start scenarios |

The key insight: **sktime is not a model library -- it is a meta-framework**.
It provides a single API surface (`fit`/`predict`/`update`) that delegates to
NeuralForecast (LSTMs, TCN), PyTorch Forecasting (N-BEATS, TFT), HuggingFace
(PatchTST, Chronos), and classical methods (ETS, ARIMA). You can benchmark
an LSTM against Chronos against ARIMA without touching your data pipeline.

In [16]:
if results:
    print(f"Approaches tested: {len(results)} of 6 demos")
    trainable = [r for r in results if r["Trainable"]]
    zero_shot = [r for r in results if not r["Trainable"]]
    if trainable:
        fastest = min(trainable, key=lambda r: r["Train Time (s)"])
        print(
            f"Fastest trainable: {fastest['Model']} ({fastest['Library']}) at {fastest['Train Time (s)']}s"
        )
    if zero_shot:
        print(f"Zero-shot models: {len(zero_shot)} (no training required)")

Approaches tested: 4 of 6 demos
Fastest trainable: LSTM (Raw PyTorch) at 3.7s
Zero-shot models: 1 (no training required)


## Key Takeaways

1. **sktime provides a unified `fit()`/`predict()` API** across NeuralForecast,
   PyTorch Forecasting, HuggingFace, and classical models
2. **Raw PyTorch gives full control** at the cost of ~10x more boilerplate --
   justified for custom architectures and research
3. **HuggingFace-backed sktime models** (PatchTST, Chronos) run cleanly in
   this Python 3.14 environment, while sktime's neuralforecast-backed demos
   are still blocked by `ray`
4. **Zero-shot foundation models** (Chronos) eliminate training entirely,
   providing instant baselines
5. **Do not compare MSE across rows unless target scale and evaluation
   window are identical** -- the `Target Scale` and `Evaluation Points`
   columns in the comparison table flag the mismatch. Raw PyTorch uses
   normalized targets and a long rolling-window test; the sktime/Darts
   rows use raw price levels and a single 5-step forecast. The scale gap
   alone changes MSE by orders of magnitude; the `n_eval` gap changes its
   sampling variance. Use this notebook to choose a workflow, not to rank
   accuracy
6. **Start with sktime for prototyping**, drop to raw PyTorch only when you
   need custom loss functions, architectures, or training dynamics

**Next**: See `09_foundation_models` for a deeper dive into zero-shot TSFMs.

**Book**: Section 13.7 discusses the practitioner framework for choosing tools.